In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import scipy as sp
import sklearn as sk
import xgboost as xgb

In [6]:
training_data = pd.read_csv("training_target_encoded.csv")
X_train = training_data.drop('target', axis=1)
y_train = training_data['target']

testing_data = pd.read_csv("testing_target_encoded.csv")
X_test = testing_data.drop('target', axis=1)
y_test = testing_data['target']

In [7]:
X_train.head()

,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,square_feet,latitude,longitude,source
0,-0.457650,-0.457650,-0.212753,-0.616307,-0.457650,0.89990,1.184478,1.630748,1.630748,-0.457650
1,-0.457650,-0.457650,1.037792,-0.616307,-0.457650,-1.10229,0.443302,-0.613215,-0.613215,-0.457650
2,-0.457650,-0.457650,-0.197154,1.622568,-0.457650,0.91440,0.810521,-0.613215,-0.613215,-0.457650
3,-0.457650,-0.457650,-0.212753,-0.616307,-0.457650,0.89990,-1.015466,1.630748,1.630748,-0.457650
4,2.185078,2.185078,1.077041,-0.616307,2.185078,-1.10229,-0.345039,-0.613215,-0.613215,2.185078


In [8]:
# Implement LASSO Regression and Cross Validate

lasso = sk.linear_model.Lasso()
hyperparams = np.linspace(1,10,1000)

lasso_crossval = sk.model_selection.GridSearchCV(lasso, param_grid={'alpha': hyperparams}, cv=5, scoring='neg_mean_squared_error')
lasso_crossval.fit(X_train, y_train)

print(f"Optimal LASSO Hyperparameter: {lasso_crossval.best_params_['alpha']}")
print(f"Best CV MSE Value {-lasso_crossval.best_score_}")

lasso_best = sk.linear_model.Lasso(alpha=lasso_crossval.best_params_['alpha'])
lasso_best.fit(X_train, y_train)

lasso_prediction = lasso_best.predict(X_test)
lasso_MSE = sk.metrics.mean_squared_error(y_test, lasso_prediction)

print(f'Inference MSE is {lasso_MSE}')
print(f'Inference RMSE is {np.sqrt(lasso_MSE)}')

Optimal LASSO Hyperparameter: 1.018018018018018
Best CV MSE Value 190954.21120999014
Inference MSE is 199920.2423542986
Inference RMSE is 447.1244148492661


In [10]:
# Repeat for XGBoost
hyperparams = {'n_estimators': [200,300,400,500],
                'max_depth': [2,4,6], 
               'learning_rate': [.005,.01,.05,.1], 
               'subsample': [.5,.6,.7,.8, .9, 1], 
               'colsample_bytree': [.7,.8,.9,1]}

xgboost_reg = xgb.XGBRegressor(n_jobs=-1)

xgboost_crossval = sk.model_selection.GridSearchCV(xgboost_reg,
                                                  param_grid=hyperparams,
                                                  cv=5,
                                                  scoring='neg_mean_squared_error',
                                                  n_jobs=-1,
                                                  verbose=1)
xgboost_crossval.fit(X_train, y_train)

Fitting 5 folds for each of 1152 candidates, totalling 5760 fits


GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None...
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=-1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.7, 0.8, 0.9, 1],
                         'learning_rate': [0.005, 0.01, 0.05, 0.1],
                         'max_depth': [2, 4, 6],
                         'n_estimators': [200, 300, 400, 500],
                         'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1]},
             scoring='neg_mean_squared_error', verbose=1)

In [11]:
print('Optimal XGBoost Hyperparameters:')
for parameter, value in xgboost_crossval.best_params_.items():
    print(f"Best {parameter} = {value}")

print()
print(f"Best CV MSE Value {-xgboost_crossval.best_score_}")

xgbest = xgboost_crossval.best_estimator_
xgboost_prediction = xgbest.predict(X_test)
xgb_MSE = sk.metrics.mean_squared_error(y_test, xgboost_prediction)

print(f'Inference MSE is {xgb_MSE}')
print(f'Inference RMSE is {np.sqrt(xgb_MSE)}')

Optimal XGBoost Hyperparameters:
Best colsample_bytree = 0.7
Best learning_rate = 0.05
Best max_depth = 2
Best n_estimators = 400
Best subsample = 0.5

Best CV MSE Value 178674.57367495616
Inference MSE is 196165.0715471944
Inference RMSE is 442.90526249661383


In [38]:
# Given that XGBoost has lower CV MSE than LASSO, we will move forward with XGBoost
# We are going to append the residuals to the testing_data, and then, we are going to run GMM.
# After, we will run statistical analysis to see if there are systematic over/under predicitons (valuations)

test_residual = testing_data.copy()
test_residual['residual'] = y_test - xgboost_prediction

# Cluster time! (we find the optimal number of clusters a few times and take the most common value!)
optimal_list = []
for i in range(20):
    BIC = []
    clusters = [i+1 for i in range(12)]
    
    for cluster in clusters:
        gmm = sk.mixture.GaussianMixture(n_components=cluster)
        gmm.fit(test_residual)
        BIC.append(gmm.bic(test_residual))
    
    optimal_cluster = clusters[np.argmin(BIC)]
    optimal_list.append(optimal_cluster)

best_cluster, count = sp.stats.mode(optimal_list)
best_cluster

C:\Users\thund\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Window

np.int64(5)

In [42]:
# Over repeated initializations, we find that 6 is the most common optimal number of clusters according to BIC
# The average optimal cluster amount if 5.55. To be safe, we initially selected 6 clusters.
# However, when we run the statistics (below) on 6 clusters, we find that there is some overfitting (a single data point has its own class)
# So, to correct this, we dropped from 6 to 5 clusters.

gmm_best = sk.mixture.GaussianMixture(n_components=(best_cluster-1))
test_residual['Cluster'] = gmm_best.fit_predict(test_residual)

C:\Users\thund\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [43]:
test_residual.head()

,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,square_feet,latitude,longitude,source,target,residual,Cluster
0,0.863349,0.863349,1.061729,-0.616307,0.863349,-1.102290,0.780200,-0.613215,-0.613215,0.863349,1589.0,-328.022095,3
1,-0.457650,-0.457650,-0.205327,0.500751,-0.457650,0.910116,1.140681,0.514490,0.514490,-0.457650,1572.0,-374.274536,3
2,0.863349,0.863349,1.061729,-0.616307,0.863349,-1.102290,-0.921134,-0.613215,-0.613215,0.863349,1987.0,624.265747,3
3,-0.457650,-0.457650,-0.205327,0.500751,-0.457650,0.910116,-0.159745,0.514490,0.514490,-0.457650,1490.0,32.645996,3
4,0.863349,0.863349,1.061729,-0.616307,0.863349,-1.102290,0.436564,-0.613215,-0.613215,0.863349,1765.0,96.028564,3


In [44]:
# Lets look at the cluster stats!
c_means = test_residual.groupby('Cluster')['residual'].agg(['mean', 'std', 'count'])
c_means

,mean,std,count
Cluster,,,
0,-13.438187,214.047528,8
1,886.328247,539.372098,4
2,-428.862579,406.505681,20
3,-15.907918,417.534517,218


In [45]:
# Finally, we are going ot run an ANOVA test to identify if clusters are significantly different from one another.
groups = []

for cluster, features in test_residual.groupby('Cluster'):
    groups.append(features['residual'].values)

f_stat, p_value = sp.stats.f_oneway(*groups)
print(f'ANOVA f-statistics {f_stat:.04f} and p-value {p_value}')
print('Since ANOVA is statistically significant, we know that at least one cluster has a significantly different average residual.')

ANOVA f-statistics 12.7988 and p-value 8.454877995677969e-08
Since ANOVA is statistically significant, we know that at least one cluster has a significantly different average residual.


In [46]:
# We will now run a Tukey HSD to get what we need
results = sp.stats.tukey_hsd(*groups)
print(results)

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)   -899.766     0.003 -1555.550  -243.983
 (0 - 2)    415.424     0.080   -32.561   863.410
 (0 - 3)      2.470     1.000  -383.032   387.971
 (1 - 0)    899.766     0.003   243.983  1555.550
 (1 - 2)   1315.191     0.000   728.640  1901.742
 (1 - 3)    902.236     0.000   361.901  1442.571
 (2 - 0)   -415.424     0.080  -863.410    32.561
 (2 - 1)  -1315.191     0.000 -1901.742  -728.640
 (2 - 3)   -412.955     0.000  -663.156  -162.753
 (3 - 0)     -2.470     1.000  -387.971   383.032
 (3 - 1)   -902.236     0.000 -1442.571  -361.901
 (3 - 2)    412.955     0.000   162.753   663.156



In [47]:
# Focus only on significant pairs!
sig_pairs = []

for i in range((best_cluster-1)):
    for j in range(i+1, (best_cluster-1)):
        if results.pvalue[i,j] < .05:
            sig_pairs.append((i,j,results.pvalue[i,j]))
for i, j, pvalue in sig_pairs:
    print(f'Cluster {i} and Cluster {j} are statistically different with a p-value of {pvalue}')

Cluster 0 and Cluster 1 are statistically different with a p-value of 0.0025953419281070644
Cluster 1 and Cluster 2 are statistically different with a p-value of 1.2140152194639597e-07
Cluster 1 and Cluster 3 are statistically different with a p-value of 0.00013305508321681536
Cluster 2 and Cluster 3 are statistically different with a p-value of 0.0001637072187186117


In [57]:
cluster_one = test_residual[test_residual['Cluster'] == 2]

In [58]:
cluster_one

,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,square_feet,latitude,longitude,source,target,residual,Cluster
10,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,0.237795,-0.613215,-0.613215,-0.457650,1300.0,-258.107788,2
16,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,0.672393,-0.613215,-0.613215,-0.457650,1396.0,-780.213135,2
36,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,0.254640,-0.613215,-0.613215,-0.457650,1650.0,134.780762,2
57,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,0.861056,-0.613215,-0.613215,-0.457650,1853.0,-388.273682,2
61,-0.457650,-0.457650,-0.205327,0.500751,-0.457650,0.910116,1.154157,0.514490,0.514490,-0.457650,2360.0,51.980713,2
91,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,0.557848,-0.613215,-0.613215,-0.457650,1723.0,-186.557861,2
100,0.863349,0.863349,1.061729,-0.616307,0.863349,-1.102290,1.191216,-0.613215,-0.613215,0.863349,1422.0,-1098.154297,2
105,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,0.264746,-0.613215,-0.613215,-0.457650,1585.0,-89.368408,2
121,-0.457650,-0.457650,-2.350386,-0.616307,-0.457650,-1.102290,1.120467,-0.613215,-0.613215,-0.457650,1662.0,-635.308838,2
126,-0.457650,-0.457650,-0.205327,0.500751,-0.457650,0.910116,1.238381,0.514490,0.514490,-0.457650,2335.0,115.575439,2
